In [ ]:
from transformers import pipeline

generator = pipeline(
    'text-generation',
    model = 'gpt2'
)

result = generator(
    'artificial intelligence is',
    max_new_tokens = 50
)

print(result[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=T

artificial intelligence is the future of human interaction.

The challenge facing AI is the same as that facing humans today: It's impossible to understand and understand the real world.

The most important thing to do in a human-driven AI is to get the


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
text =  'artificial intelligence is amazing'

# The 'Ġ' character indicates a space. This is a common convention in Byte Pair Encoding (BPE) tokenizers,
# like the one used for GPT-2, to differentiate between tokens that start with a space and those that don't.
# For example, ' intelligence' is different from 'intelligence' in terms of tokenization.
# Tokenizers break down text into subword units (like 'art', 'ificial') to handle out-of-vocabulary words
# and to represent common words efficiently.
tokens = tokenizer.tokenize(text)
print(tokens)

['art', 'ificial', 'Ġintelligence', 'Ġis', 'Ġamazing']


In [ ]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

[433, 9542, 4430, 318, 4998]


In [ ]:
encoded = tokenizer(text)
print(encoded)

{'input_ids': [433, 9542, 4430, 318, 4998], 'attention_mask': [1, 1, 1, 1, 1]}


In [ ]:
decoded = tokenizer.decode(encoded['input_ids'])
print(decoded)

artificial intelligence is amazing


### Example of Padding Incomplete Batches

When working with models, you often process multiple sequences (a batch) at once. These sequences might have different lengths. To feed them into a model, they need to be of a uniform length. This is where padding comes in.

We'll create a batch of tokenized texts, then use `tokenizer.pad` to ensure they all have the same length by adding `pad_token_id` and generating an `attention_mask`.

In [ ]:
# Define a few example texts of different lengths
batch_texts = [
    'This is a short sentence.',
    'This is a slightly longer sentence for demonstration purposes.',
    'Another text.'
]

# Tokenize each text in the batch
batch_tokenized = [tokenizer.encode(text) for text in batch_texts]

print("Original token IDs for each sequence:")
for i, ids in enumerate(batch_tokenized):
    print(f"Sequence {i+1} (length {len(ids)}): {ids}")

# Set the pad_token for the tokenizer. For GPT2, it's common to use the EOS token.
# This resolves the ValueError encountered previously.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Identify the pad token ID. For GPT2, it's typically the EOS token ID.
pad_token_id = tokenizer.pad_token_id
print(f"\nPad token ID: {pad_token_id}")

# Pad the batch of tokenized sequences
# padding='longest' pads to the length of the longest sequence in the batch
# return_attention_mask=True generates an attention mask
padded_batch = tokenizer.pad(
    {'input_ids': batch_tokenized},
    padding='longest',
    return_attention_mask=True,
    return_tensors='pt' # Return as PyTorch tensors for convenience
)

print("\nPadded input_ids (token IDs):")
print(padded_batch['input_ids'])

print("\nAttention mask (1 for real tokens, 0 for padding tokens):")
print(padded_batch['attention_mask'])

print("\nDecoded padded input IDs to show the original and padding:")
for i, input_ids in enumerate(padded_batch['input_ids']):
    print(f"Sequence {i+1}: {tokenizer.decode(input_ids, skip_special_tokens=False)}")

Original token IDs for each sequence:
Sequence 1 (length 6): [1212, 318, 257, 1790, 6827, 13]
Sequence 2 (length 10): [1212, 318, 257, 4622, 2392, 6827, 329, 13646, 4959, 13]
Sequence 3 (length 3): [6610, 2420, 13]

Pad token ID: 50256

Padded input_ids (token IDs):
tensor([[ 1212,   318,   257,  1790,  6827,    13, 50256, 50256, 50256, 50256],
        [ 1212,   318,   257,  4622,  2392,  6827,   329, 13646,  4959,    13],
        [ 6610,  2420,    13, 50256, 50256, 50256, 50256, 50256, 50256, 50256]])

Attention mask (1 for real tokens, 0 for padding tokens):
tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]])

Decoded padded input IDs to show the original and padding:
Sequence 1: This is a short sentence.<|endoftext|><|endoftext|><|endoftext|><|endoftext|>
Sequence 2: This is a slightly longer sentence for demonstration purposes.
Sequence 3: Another text.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftex

As you can see:
- The `input_ids` for shorter sentences have been extended with the `pad_token_id` (50256).
- The `attention_mask` has `1` for original tokens and `0` for padding tokens. This mask tells the model to ignore the padding tokens during computation.
- When decoding, you can either skip special tokens (including padding) or observe them, depending on the `skip_special_tokens` argument.

## Understanding Transformer Architectures: Encoder-Only, Decoder-Only, and Encoder-Decoder Models

Transformer models are the backbone of many modern AI applications, especially in Natural Language Processing (NLP). They come in different flavors, each designed for specific tasks. Let's break down the three main types:

1.  **Encoder-Only Models** (e.g., BERT, RoBERTa)
2.  **Decoder-Only Models** (e.g., GPT, LLaMA, Mistral)
3.  **Encoder-Decoder Models** (e.g., T5, BART, NLLB)

### 1. Encoder-Only Models: Understanding Context

#### What they are and why they are so:

Imagine you have a jigsaw puzzle, and your goal is to understand the full picture, not to add new pieces. Encoder-only models are like highly skilled puzzle solvers. Their main job is to **understand the context** of every piece (word/token) in a given sentence or text. They look at all words simultaneously, both left and right, to grasp the full meaning.

Their architecture consists of multiple "encoder" blocks stacked on top of each other. Each block takes a sequence of token embeddings, processes them, and outputs a refined set of embeddings that capture richer contextual information for each token.

#### How they are trained:

Encoder-only models are typically trained using two main self-supervised tasks:

1.  **Masked Language Modeling (MLM)**: The model is given a sentence where some words are randomly hidden (masked). Its task is to predict the original masked words. For example, if the sentence is "The quick brown [MASK] jumps over the lazy [MASK]", the model learns to predict "fox" and "dog" by understanding the surrounding context.
2.  **Next Sentence Prediction (NSP)**: The model is given two sentences and has to predict if the second sentence logically follows the first. This helps the model understand relationships between sentences.

These training methods force the model to learn deep contextual representations of words.

#### How they are used and why only used for that applications:

Because encoder-only models excel at understanding the meaning of text, they are perfect for tasks where you need to classify, analyze, or extract information from existing text. They don't generate new text but rather provide a rich understanding of the input.

**Typical Applications:**

*   **Sentiment Analysis:** Is this review positive or negative?
*   **Text Classification:** Is this email spam or not spam?
*   **Named Entity Recognition (NER):** Identify people, organizations, locations in a text.
*   **Question Answering ( extractive QA):** Given a paragraph and a question, pinpoint the exact answer within the paragraph.
*   **Search and Information Retrieval:** Understanding queries and documents to find the most relevant results.

**Why these applications?** These tasks require a deep, bidirectional understanding of the input text. The model doesn't need to generate new words; it just needs to "read" and "comprehend" what's already there.

### 2. Decoder-Only Models: Generating Text

#### What they are and why they are so:

If encoder-only models are puzzle solvers, decoder-only models are creative storytellers. Their primary function is to **generate new text**, one token at a time, based on the input they've already received. They are designed to predict the *next* word in a sequence.

Their architecture consists of multiple "decoder" blocks. A key characteristic is the "masked self-attention" mechanism, which ensures that when predicting the current word, the model can *only* look at the words that came before it, not the words that come after (which haven't been generated yet). This is crucial for text generation to maintain coherence and flow.

#### How they are trained:

Decoder-only models are trained on a simpler but massive scale: **Language Modeling**. This means they are given a vast amount of text and are trained to predict the next word in a sequence. For example, if the input is "The cat sat on the", the model learns to predict "mat".

By repeatedly predicting the next word, the model learns grammar, syntax, facts, and even common sense from the training data. It learns to complete sentences and generate coherent passages.

#### How they are used and why only used for that applications:

Because decoder-only models are experts at predicting the next token, they are ideal for any task that involves generating free-form text. They are the engine behind most conversational AIs and creative writing tools.

**Typical Applications:**

*   **Text Generation:** Writing articles, stories, poems, or code.
*   **Chatbots and Conversational AI:** Responding to user queries in a human-like way.
*   **Creative Writing:** Assisting authors with plot ideas or sentence suggestions.
*   **Summarization (abstractive):** Generating a new summary that wasn't directly in the original text.
*   **Translation (less common for traditional NMT but possible):** Generating a translated sentence word-by-word.

**Why these applications?** These tasks fundamentally require creating novel text. The model needs to "imagine" or infer what comes next, following the patterns it learned during its massive language modeling training.

### 3. Encoder-Decoder Models: Translation and Transformation

#### What they are and why they are so:

Encoder-decoder models are like expert interpreters. They take an input in one "language" (or format), fully understand it, and then generate an output in a different "language" (or format). They combine the strengths of both encoders and decoders.

Their architecture has two main parts:

1.  **An Encoder**: This part takes the entire input sequence, reads it bidirectionally (like an encoder-only model), and builds a rich, contextual understanding of the *source* text.
2.  **A Decoder**: This part takes the contextual understanding from the encoder and generates the *target* output sequence one token at a time (like a decoder-only model, using masked self-attention).

The key connection between them is an "encoder-decoder attention" mechanism, which allows the decoder to focus on relevant parts of the encoded input when generating each word of the output.

#### How they are trained:

Encoder-decoder models are typically trained on "sequence-to-sequence" tasks. This means they are given an input sequence and taught to transform it into a different output sequence. Common training tasks include:

*   **Machine Translation:** Given a sentence in English, output the same sentence in French.
*   **Summarization:** Given a long document, output a shorter summary.
*   **Text Simplification:** Given complex text, output simpler text.

#### How they are used and why only used for that applications:

These models are ideal for tasks that involve transforming one sequence of text into another, often changing its form, language, or length. They need to understand the entire input to create a relevant output.

**Typical Applications:**

*   **Machine Translation:** The classic use case, translating text from one language to another.
*   **Abstractive Summarization:** Generating a concise summary from a longer text.
*   **Dialogue Generation:** Generating responses in a conversation where the input is the conversation history.
*   **Data-to-Text Generation:** Converting structured data (e.g., a weather report's numbers) into natural language descriptions.

**Why these applications?** These tasks involve a clear distinction between an input that needs to be fully understood (handled by the encoder) and an output that needs to be generated sequentially (handled by the decoder), with a strong focus on transforming information from one form to another.

### Summary

| Model Type       | Focus / Primary Goal           | Information Flow         | Key Mechanism       | Typical Applications                                    |
| :--------------- | :----------------------------- | :----------------------- | :------------------ | :------------------------------------------------------ |
| **Encoder-Only** | Understanding input context    | Bidirectional (looks left & right) | Bidirectional Self-Attention | Classification, NER, Extractive QA, Sentiment Analysis  |
| **Decoder-Only** | Generating new text            | Unidirectional (looks left only)  | Masked Self-Attention | Text Generation, Chatbots, Creative Writing, Abstractive Summarization |
| **Encoder-Decoder** | Transforming input to output   | Encoder: Bidirectional; Decoder: Unidirectional with Encoder-Decoder Attention | Encoder-Decoder Attention | Machine Translation, Abstractive Summarization, Data-to-Text |

Each architecture is a powerful tool, specialized for different stages of language processing. Understanding their strengths helps in choosing the right model for your specific NLP task!

## The Lifecycle of an LLM: Training and Inference

Every large language model goes through two primary phases: **training** and **inference**. These are distinct but equally important steps in bringing an LLM to life and putting it to work.

### The Training Phase: Learning from the World

#### What happens in Training?

Think of the training phase as the LLM's education. This is where the model learns everything it knows about language, facts, reasoning, and even some creativity. It's an incredibly resource-intensive process that involves:

1.  **Massive Datasets**: LLMs are trained on truly enormous datasets of text and code. This includes books, articles, websites, conversations, scientific papers, source code, and much more. These datasets can span terabytes of information.
2.  **Self-Supervised Learning**: As discussed with encoder-only and decoder-only models, the training is often *self-supervised*. This means the data itself provides the supervision. For example, the model predicts masked words (MLM) or the next word in a sequence (Language Modeling). It doesn't require humans to manually label examples.
3.  **Parameter Tuning**: The neural network within the LLM has billions (or even trillions) of adjustable parameters (weights and biases). During training, the model processes the vast dataset, makes predictions, compares its predictions to the actual data, and then adjusts these parameters incrementally to improve its accuracy. This process is driven by complex optimization algorithms.
4.  **Hardware**: Training requires immense computational power, typically using hundreds or thousands of specialized GPUs (Graphics Processing Units) running for weeks or months.

#### How Models Get Trained:

*   **Pre-training**: This is the initial, large-scale, self-supervised training phase where the model learns general language understanding and generation capabilities from diverse internet text.
*   **Fine-tuning (and Alignment)**: After pre-training, models often undergo further, more targeted training. This can include:
    *   **Supervised Fine-tuning (SFT)**: Training on smaller, high-quality, human-curated datasets for specific tasks (e.g., following instructions, answering questions). This helps the model become more useful and aligned with human intent.
    *   **Reinforcement Learning from Human Feedback (RLHF)**: Humans rank different model outputs, and this feedback is used to further refine the model's behavior, making it more helpful, harmless, and honest. This is a critical step for aligning models with human values and preferences.

**Analogy**: Pre-training is like a child reading every book in a library. Fine-tuning and alignment are like that child then attending a specialized school, learning to apply their knowledge, and being taught manners and helpful behavior.

### The Inference Phase: Putting Knowledge to Work

#### What happens in Inference?

The inference phase is when the trained LLM is actually *used* to perform tasks. This is what happens every time you interact with a chatbot, ask a question to an AI assistant, or generate text.

1.  **Input Processing**: The user's input (prompt) is tokenized and converted into numerical IDs, just like during training.
2.  **Forward Pass**: These token IDs are fed through the *already trained* neural network. The model performs calculations using its fixed, learned parameters to produce an output.
3.  **Output Generation**: For generative models (like decoder-only or encoder-decoder), the model predicts the most likely next token, then the next, and so on, until it generates a complete response (or reaches a stopping condition).
4.  **Decoding**: The generated token IDs are converted back into human-readable text.

#### How Inference Models are Used:

Unlike training, inference is about *applying* the learned knowledge. It's generally much less computationally expensive than training (though still requires significant resources for large models). The model's parameters are fixed; it's not learning anything new during this phase (unless it's a specific form of online learning, which is less common for foundational LLMs).

**Use Cases**: All the applications we discussed for encoder-only, decoder-only, and encoder-decoder models fall under the inference phase. Whether it's classifying text, generating an email, or translating a document, it's all inference.

## What's Going On in the World of LLMs Today?

The field of Large Language Models is exploding with innovation! Here are some key trends and developments:

1.  **Scaling Up**: Models continue to grow in size (number of parameters) and are trained on even larger datasets. This often leads to improved performance and emergent capabilities.
2.  **Multimodality**: LLMs are moving beyond just text. Models are increasingly being trained to understand and generate content across different modalities, such as images, audio, and video (e.g., GPT-4o, Gemini).
3.  **Efficiency and Optimization**: A huge area of research is making LLMs smaller, faster, and more efficient for training and inference. Techniques like quantization, pruning, and distillation are crucial for deploying models on less powerful hardware or in real-time applications.
4.  **Agentic AI**: LLMs are being equipped with tools and the ability to plan and act. Instead of just generating text, they can browse the web, use APIs, perform calculations, and interact with other software to achieve complex goals.
5.  **Specialization and Customization**: While general-purpose models are powerful, there's a growing trend towards specialized LLMs that are fine-tuned for particular domains (e.g., legal, medical) or tasks, often achieving higher accuracy in those specific areas.
6.  **Open Source vs. Proprietary**: There's a dynamic tension between large proprietary models (e.g., OpenAI's GPT series, Google's Gemini) and a vibrant open-source community (e.g., LLaMA, Mistral, Gemma) that is pushing innovation and accessibility.
7.  **Ethical AI and Safety**: Significant effort is being put into addressing bias, fairness, transparency, and safety concerns. This includes developing better alignment techniques, robust moderation systems, and methods to detect and mitigate harmful outputs.
8.  **Longer Context Windows**: Models are capable of processing and remembering much larger amounts of input text, allowing for more complex conversations and analysis of longer documents.

It's a rapidly evolving field, constantly pushing the boundaries of what AI can do with language!

That's a great question that gets to the core of how language models process information! You're right that the 'semantic meaning' is primarily captured in the embeddings (the dense vectors) that are generated for each token. However, the token IDs are a crucial and necessary intermediate step:

Numerical Representation for Computers: Computers and neural networks work with numbers, not text directly. Tokenization converts human-readable words (or subwords) into a numerical format that the machine can understand.

Indexing Embedding Tables: Each unique token in a model's vocabulary is assigned a unique integer ID. These IDs act as indices into an "embedding table" (or embedding layer) within the neural network. When a model processes an input sequence, it uses these token IDs to look up the corresponding pre-trained or learned embedding vector for each token.

Efficiency: Using integer IDs is much more efficient computationally than processing string tokens directly. It allows for fast lookup and vectorized operations.

Input to Models: The input_ids array (which contains these token IDs) is the primary input that goes into a Transformer model. The very first layer of a Transformer model is typically an embedding layer, which takes these input_ids and converts them into the dense semantic vectors (embeddings) that subsequent layers will then process to understand context and generate meaning.

So, while the embeddings are where the rich semantic information lies, the token IDs are the essential numerical keys that unlock and access those embeddings, allowing the model to work with text in a numerical format. Think of it like a book in a library: the token_id is the call number, and the embedding is the actual book containing all the information.